# 0.4 Xatu-Direct BAL RLP Constructor

This notebook builds a block-level raw RLP BAL from Xatu tables. It is section-based, not count-based: storage writes, storage reads, balance diffs, nonce diffs, code diffs, and touched accounts are grouped by account and then RLP encoded.

The output is suitable for the bandwidth simulator as `bal_bytes`, with a calibration caveat: public Xatu storage-read coverage may not exactly match the RPC `prestateTracer` read set used by `eth-bal-analysis`.

## Why BAL Has These Sections

BAL is account-centric. Each account entry is encoded as:

```text
[address, storage_writes, storage_reads, balance_changes, nonce_changes, code_changes]
```

- `storage_writes`: slots whose post value changed.
- `storage_reads`: read-only slots, after excluding slots written in the same block.
- `balance_changes`: ETH balance post-values after transfers, gas accounting, withdrawals, and other balance effects.
- `nonce_changes`: sender nonce increments and creation-related nonce changes.
- `code_changes`: deployed code bytes for contract creations.
- touched accounts with empty sections still cost bytes because the account address is in the BAL.

So bandwidth is not only calldata plus storage slots. It is calldata plus the raw encoded BAL payload.

In [ ]:
import os
from pathlib import Path

import clickhouse_connect
import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"notebooks", "archived"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
sys.path.insert(0, str(PROJECT_ROOT / "archived"))

from xatu_bal import build_xatu_bal_for_block

load_dotenv(PROJECT_ROOT / ".env")
missing = [name for name in ["CLICKHOUSE_USER", "CLICKHOUSE_PASSWORD"] if not os.environ.get(name)]
if missing:
    raise RuntimeError("Missing ClickHouse credentials: " + ", ".join(missing))

client = clickhouse_connect.get_client(
    host="clickhouse-raw.xatu.ethpandaops.io",
    port=443,
    secure=True,
    username=os.environ["CLICKHOUSE_USER"],
    password=os.environ["CLICKHOUSE_PASSWORD"],
)
print(client.query("SELECT version()").result_rows)

In [ ]:
NETWORK = "mainnet"
BLOCKS = [22_886_891]

# read_not_written matches BAL builder semantics. all_xatu_reads is a diagnostic upper variant
# for the reads Xatu exposes. none is the no-reads sensitivity.
READ_MODES = ["read_not_written", "all_xatu_reads", "none"]

# balance_reads matches the account count for the 22_886_891 eth-bal-analysis sample better
# than address_appearances, which can include indexer relationships outside the BAL shell.
TOUCHED_ACCOUNT_MODE = "balance_reads"
WRITE_CSV = True

In [ ]:
rows = []
for block_number in BLOCKS:
    for read_mode in READ_MODES:
        result = build_xatu_bal_for_block(
            client,
            block_number=block_number,
            network=NETWORK,
            read_mode=read_mode,
            touched_account_mode=TOUCHED_ACCOUNT_MODE,
        )
        rows.append(result.summary.as_dict())

summary = pd.DataFrame(rows)
display(summary)

if WRITE_CSV:
    out = PROJECT_ROOT / "markdowns" / f"xatu_direct_bal_summary_{min(BLOCKS)}_{max(BLOCKS)}.csv"
    summary.to_csv(out, index=False)
    print(out)

In [ ]:
# Optional local calibration against nerolation/eth-bal-analysis raw RLP samples.
# This cell does not require RPC; it only checks files if you have the repo cloned locally.

sample_dir = Path("/private/tmp/eth-bal-analysis/bal_raw/rlp")
calibration_rows = []
if sample_dir.exists():
    for block_number in BLOCKS:
        sample = sample_dir / f"{block_number}_with_reads.rlp"
        if sample.exists():
            sample_bytes = sample.stat().st_size
            for _, row in summary[summary["block_number"] == block_number].iterrows():
                calibration_rows.append({
                    "block_number": block_number,
                    "read_mode": row["read_mode"],
                    "xatu_bal_rlp_bytes": int(row["bal_rlp_bytes"]),
                    "sample_bal_rlp_bytes": sample_bytes,
                    "delta_bytes": int(row["bal_rlp_bytes"]) - sample_bytes,
                })

calibration = pd.DataFrame(calibration_rows)
display(calibration)